splitting the job into two layered work:

1 - fast_pass_scan: 

2 - deep scan of the leftovers

In [1]:
# -*- coding: utf-8 -*-
import os
import re
import pandas as pd
from typing import List, Pattern, Tuple, Dict, Any

# === CONFIGURATION ===
CONFIG_DIR = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files"
OUTPUT_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\All_YMLs\3.1_YML_Files.csv"
OUTDIR = os.path.dirname(OUTPUT_CSV)
os.makedirs(OUTDIR, exist_ok=True)

MISSES_CSV = os.path.join(OUTDIR, "3.1_YML_Missed_Triggers_Audit.csv")
MISSES_SUMMARY_CSV = os.path.join(OUTDIR, "3.1_YML_Missed_Triggers_Summary.csv")

Search_Method_Name = (
    "Stage 1 (fast) - normalize run/script + multi-module + GMD deviceCheck + "
    "generic *AndroidTest (excludes) + var-hinted gradle + stronger GHA inputs + "
    "Spoon/Marathon/Flank + audit v2"
)

# --- helpers ---
def compile_any(patterns: List[str], flags=re.I | re.M) -> List[Pattern]:
    return [re.compile(p, flags) for p in patterns]

def any_match(patterns: List[Pattern], text: str) -> bool:
    return any(p.search(text) for p in patterns)

def unique_preserve(seq: List[str]) -> List[str]:
    seen, out = set(), []
    for x in seq:
        if x not in seen:
            seen.add(x); out.append(x)
    return out

COMMENT_LINE_RE = re.compile(r'(?m)^\s*(#|//|REM\b|::).*?$')
def strip_comments(raw: str) -> str:
    return COMMENT_LINE_RE.sub("", raw or "")

# === NORMALIZATION ===
def normalize_block_keys(text: str) -> str:
    # Single-line: "run: ./gradlew ..." -> "./gradlew ..."
    text = re.sub(r'(?mi)^\s*(script|run|command)\s*:\s*(?!\|)\s*(.+)$', r'\2', text)
    # Multiline: drop the key line only (keep content)
    text = re.sub(r'(?mi)^\s*(script|run|command)\s*:\s*\|?\s*$', '', text)
    return text

# Gradle prefix: env=..., sudo, (bash|sh) -c, pre-commands with &&, cd &&, gradle/gradlew wrapper
GRADLE_PREFIX = (
    r'^\s*'
    r'(?:\S+=\S+\s+)*'
    r'(?:sudo\s+)?'
    r'(?:(?:bash|sh)\s+-c[l]?\s+[\'"]?)?'
    r'(?:[^#\n;]*?&&\s+)?'
    r'(?:cd\s+\S+\s+&&\s+)?'
    r'(?:\./|\.\\)?gradle(?:w)?(?:\.bat)?'
)
GRADLE_ANYWHERE = r'(?i)[^\n]*\bgradle(?:w)?(?:\.bat)?[^\n]*'

# === DEVICE SOURCES (robust & fast) ===
DEVICE_SOURCES = [
    # Real devices
    ("Real_Device", "adb devices",      [r'(?m)^\s*adb\s+devices\b']),
    ("Real_Device", "adb get-state",    [r'(?m)^\s*adb\s+get-state\b']),
    ("Real_Device", "adb get-serialno", [r'(?m)^\s*adb\s+get-serialno\b']),
    ("Real_Device", "adb -s <serial> (physical)", [r'(?m)^\s*adb\s+-s\s+(?!emulator-\d+\b)(?!localhost:\d+\b)(?!127\.0\.0\.1:\d+\b)\S+\b']),
    ("Real_Device", "adb install",      [r'(?m)^\s*adb\s+install(\s+-r)?\b']),
    ("Real_Device", "adb shell",        [r'(?m)^\s*adb\s+shell\b']),
    ("Real_Device", "adb root",         [r'(?m)^\s*adb\s+root\b']),
    ("Real_Device", "adb settings",     [r'(?m)^\s*adb\s+shell\s+settings\b']),
    ("Real_Device", "adb input",        [r'(?m)^\s*adb\s+shell\s+input\b']),
    ("Real_Device", "adb pm grant",     [r'(?m)^\s*adb\s+shell\s+pm\s+grant\b']),

    # Emulator signals
    ("Emulator", "adb -s emulator-serial", [
        r'(?m)^\s*adb\s+-s\s+emulator-\d+\b',
        r'(?m)^\s*adb\s+-s\s+(?:localhost|127\.0\.0\.1):\d+\b',
    ]),
    ("Emulator", "emulator -avd/@", [r'(?m)^\s*\S*emulator\b[^\n]*\s(-avd|@)\S+']),
    ("Emulator", "android-wait-for-emulator", [r'(?m)^\s*(?:\./)?android-wait-for-emulator\b']),
    ("Emulator", "start-emulator.sh", [r'(?m)^\s*start-emulator\.sh\b']),
    ("Emulator", "android create avd", [r'(?m)^\s*\S*android\b[^\n]*\bcreate\s+avd\b']),
    ("Emulator", "circle-android wait-for-boot",[r'(?m)^\s*circle-android\s+wait-for-boot\b']),
    ("Emulator", "reactivecircus runner", [r'uses:\s*reactivecircus/android-emulator-runner']),
    ("Emulator", "sys-img component", [
        r'(?m)^\s*-\s*sys-img-[^\s]*-android-(?:\d+|\$[A-Z_][A-Z0-9_]*)\b',
        r'(?m)^\s*-\s*sys-img-[^\s]*-google_apis-[^\s]*(?:\d+|\$[A-Z_][A-Z0-9_]*)\b'
    ]),
    ("Emulator", "avdmanager", [r'(?m)^\s*\S*avdmanager\b']),
    ("Emulator", "sdkmanager system-images/emulator", [
        r'(?m)^\s*\S*sdkmanager\b[^\n"]*"system-images;android-(?:\d+|\$[A-Z_][A-Z0-9_]*)[^"\n]*"|'
        r'(?m)^\s*\S*sdkmanager\b[^\n]*\bsystem-images;android-(?:\d+|\$[A-Z_][A-Z0-9_]*)\b'
    ]),
    ("Emulator", "api-level", [r'\bapi[-_ ]?level\b\s*:?\s*\d{2}']),
    ("Emulator", "abi/arch",  [r'\b(abi|arch)\b\s*:?\s*(x86|x86_64|arm64|armeabi)']),
    ("Emulator", "target image", [r'\btarget\s*:\s*(google_apis|google_apis_playstore|aosp.*)']),
    ("Emulator", "device name", [r'\b(avd[-_ ]?name|device)\b\s*:\s*pixel']),

    # Gradle Managed Devices (GMD)
    ("GMD", "managedDevices DSL",       [r'\bmanageddevices?\b']),
    ("GMD", "ManagedVirtualDevice DSL", [r'\bmanagedvirtualdevice\b|\bcom\.android\.build\.api\.dsl\.ManagedVirtualDevice\b']),
    ("GMD", "GMD task mentions",        [r'\bmanageddevice\w*androidtest\b']),
    ("GMD", "GHA gradle arguments/tasks", [
        r'(?m)^\s*arguments\s*:\s*[:\w-]*manageddevice\w*androidtest\b',
        r'(?m)^\s*tasks?\s*:\s*[:\w-]*manageddevice\w*androidtest\b'
    ]),

    # Third-party device labs
    ("Third_Party_Lab", "gcloud firebase", [r'(?m)^\s*gcloud(\s+beta)?\s+firebase\s+test\s+android\s+run\b']),
    ("Third_Party_Lab", "saucectl",        [r'(?m)^\s*saucectl(\s+run)?\b']),
    ("Third_Party_Lab", "browserstack/bstack",[r'\b(browserstack|bstack)\b']),
    ("Third_Party_Lab", "appcenter test",  [r'(?m)^\s*appcenter\s+test\s+run\s+android\b']),
    ("Third_Party_Lab", "maestro cloud",   [r'(?m)^\s*maestro\s+cloud\b']),
    ("Third_Party_Lab", "test_matrix/firebase.json", [r'\btest_matrix\.json\b|\bfirebase\.json\b']),
]

# === TRIGGERS (expanded & fast) ===
NON_TEST_PREFIX = r'(?:assemble|bundle|package|compile|merge|process|generate|install|uninstall|jacoco|lint|publish|sign|upload)'

TRIGGER_SOURCES = [
    # Connected
    ("Gradle", "connectedAndroidTest",                 [rf'(?m){GRADLE_PREFIX}[^\n]*\bconnectedandroidtest\b']),
    ("Gradle", "connected.*Android.*",                 [rf'(?m){GRADLE_PREFIX}[^\n]*\bconnected[a-z0-9:._-]*android[a-z0-9:._-]*test\b']),
    ("Gradle", "connectedCheck",                       [rf'(?m){GRADLE_PREFIX}\s+(?::[\w-]+:)*connectedcheck\b']),
    ("Gradle", "connectedAndroidTest (abbr)",          [rf'(?m){GRADLE_PREFIX}[^\n]*\b(?:cat|connectedandroidtest)\b']),

    # Managed devices / aggregators
    ("Gradle", "deviceCheck",                          [rf'(?mi){GRADLE_PREFIX}\s+(?::[\w-]+:)*(?:devicecheck|alldevicechecks)\b']),
    ("Gradle", "managedDevice AndroidTest",            [rf'(?mi){GRADLE_PREFIX}[^\n]*\b(?!{NON_TEST_PREFIX})(?:manageddevice|device)[\w:-]*androidtest\b']),

    # Generic *AndroidTest variant/device tasks (exclude non-test verbs)
    ("Gradle", "variant/device AndroidTest",           [rf'(?mi){GRADLE_PREFIX}[^\n]*\b(?!{NON_TEST_PREFIX})[\w:-]*androidtest\b']),
    ("Gradle", "plain androidTest",                    [rf'(?mi){GRADLE_PREFIX}[^\n]*\b(?::[\w-]+:)*androidtest\b']),

    # Third-party runners & CLIs (Gradle tasks)
    ("Gradle", "Spoon",                                [rf'(?mi){GRADLE_PREFIX}[^\n]*\bspoon(?:\w*androidtest)?\b']),
    ("Gradle", "Marathon",                             [rf'(?mi){GRADLE_PREFIX}[^\n]*\bmarathon(?:\w*androidtest)?\b']),

    # Anywhere-on-line fallbacks
    ("Gradle", "connected (anywhere)",                 [rf'(?mi){GRADLE_ANYWHERE}\bconnected[a-z0-9:._-]*android[a-z0-9:._-]*test\b']),
    ("Gradle", "connectedAndroidTest (abbr, anywhere)",[rf'(?mi){GRADLE_ANYWHERE}\b(?:cat|connectedandroidtest)\b']),
    ("Gradle", "deviceCheck (anywhere)",               [rf'(?mi){GRADLE_ANYWHERE}\b(?::[\w-]+:)*(?:devicecheck|alldevicechecks)\b']),
    ("Gradle", "managedDevice AndroidTest (anywhere)", [rf'(?mi){GRADLE_ANYWHERE}\b(?!{NON_TEST_PREFIX})(?:manageddevice|device)[\w:-]*androidtest\b']),
    ("Gradle", "variant/device AndroidTest (anywhere)",[rf'(?mi){GRADLE_ANYWHERE}\b(?!{NON_TEST_PREFIX})[\w:-]*androidtest\b']),
    ("Gradle", "Spoon (anywhere)",                     [rf'(?mi){GRADLE_ANYWHERE}\bspoon(?:\w*androidtest)?\b']),
    ("Gradle", "Marathon (anywhere)",                  [rf'(?mi){GRADLE_ANYWHERE}\bmarathon(?:\w*androidtest)?\b']),

    # Other venues (non-gradle)
    ("ADB", "am instrument",                           [r'(?mi)^[^\n]*\bam\s+instrument\b']),
    ("Third_Party_Lab", "gcloud firebase",             [r'(?mi)^[^\n]*\bgcloud(?:\s+beta)?\s+firebase\s+test\s+android\s+run\b']),
    ("Third_Party_Lab", "flank",                       [r'(?mi)^[^\n]*\bflank\s+android\s+run\b']),
    ("Third_Party_Lab", "saucectl",                    [r'(?mi)^[^\n]*\bsaucectl(?:\s+run)?\b']),
    ("Third_Party_Lab", "appcenter run",               [r'(?mi)^[^\n]*\bappcenter\s+test\s+run\s+android\b']),
    ("Flutter", "flutter drive",                       [r'(?mi)^[^\n]*\bflutter\s+drive\b']),
    ("Flutter", "flutter test (integration_test)",     [r'(?mi)^[^\n]*\bflutter\s+test\b[^\n]*\bintegration_test\b']),
    ("Flutter", "dart test (integration_test)",        [r'(?mi)^[^\n]*\bdart\s+test\b[^\n]*\bintegration_test\b']),
]

# gradle/gradle-build-action inputs (expanded)
GHA_GRADLE_INPUTS = compile_any([
    r'(?mi)^\s*arguments\s*:\s*[:\w-]*connected.*android.*test\b',
    r'(?mi)^\s*arguments\s*:\s*(?::[\w-]+:)*connectedcheck\b',
    r'(?mi)^\s*arguments\s*:\s*(?::[\w-]+:)*(?:devicecheck|alldevicechecks)\b',
    r'(?mi)^\s*arguments\s*:\s*[\w:-]*androidtest\b',
    r'(?mi)^\s*arguments\s*:\s*\bcat\b',

    r'(?mi)^\s*tasks?\s*:\s*[:\w-]*connected.*android.*test\b',
    r'(?mi)^\s*tasks?\s*:\s*(?::[\w-]+:)*connectedcheck\b',
    r'(?mi)^\s*tasks?\s*:\s*(?::[\w-]+:)*(?:devicecheck|alldevicechecks)\b',
    r'(?mi)^\s*tasks?\s*:\s*[\w:-]*androidtest\b',
    r'(?mi)^\s*tasks?\s*:\s*\bcat\b',
])

# Variable-hinted gradle lines
VAR_HINT = re.compile(r'\${{\s*(?:matrix|env|vars|inputs)\.([^}]+)\s*}}', re.I)
def variable_hints_connected(line: str) -> bool:
    m = VAR_HINT.search(line)
    if not m:
        return False
    hint = m.group(1).lower()
    return any(k in hint for k in [
        "connected","androidtest","device","managed","e2e","ui","espresso","instrument"
    ])

# Precompile pattern sets
DEVICE_PATTERNS  = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in DEVICE_SOURCES]
TRIGGER_PATTERNS = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in TRIGGER_SOURCES]

def collect_hits_with_groups(patterns: List[Tuple[str, str, List[Pattern]]], text: str):
    labels, groups = [], []
    for grp, lbl, pats in patterns:
        if any_match(pats, text):
            labels.append(lbl); groups.append(grp)
    return unique_preserve(labels), unique_preserve(groups)

# === WEAK-HINT GATING & RECONCILIATION ===
STRONG_DEVICE_LABELS = {
    "emulator -avd/@", "android-wait-for-emulator", "start-emulator.sh",
    "circle-android wait-for-boot", "reactivecircus runner",
    "avdmanager", "sdkmanager system-images/emulator","android create avd",
    "managedDevices DSL", "ManagedVirtualDevice DSL", "GMD task mentions", "GHA gradle arguments/tasks",
    "adb get-state", "adb get-serialno", "adb -s <serial> (physical)",
    "gcloud firebase", "saucectl", "browserstack/bstack", "appcenter test", "maestro cloud",
    "test_matrix/firebase.json",
}
WEAK_DEVICE_LABELS = {"api-level", "abi/arch", "target image", "device name", "sys-img component"}

def filter_weak_device_hints(labels, groups):
    if not (set(labels) & STRONG_DEVICE_LABELS):
        labels = [l for l in labels if l not in WEAK_DEVICE_LABELS]
        if not labels:
            groups = []
    return labels, groups

EMULATOR_STRONG_LABELS = {
    "emulator -avd/@", "android-wait-for-emulator", "start-emulator.sh",
    "circle-android wait-for-boot", "reactivecircus runner", "avdmanager",
    "sdkmanager system-images/emulator","adb -s emulator-serial"
}
REAL_DEVICE_STRONG_LABELS = {"adb get-state", "adb get-serialno", "adb -s <serial> (physical)"}
REAL_DEVICE_GENERIC_ADB = {"adb devices","adb install", "adb shell", "adb root", "adb settings", "adb input", "adb pm grant"}

def reconcile_emulator_vs_real(labels, groups):
    lbls = set(labels)
    if lbls & EMULATOR_STRONG_LABELS:
        lbls -= REAL_DEVICE_GENERIC_ADB
        labels = [l for l in labels if l in lbls]
        if "Real_Device" in groups:
            has_real_after = bool(set(labels) & REAL_DEVICE_STRONG_LABELS)
            if not has_real_after:
                groups = [g for g in groups if g != "Real_Device"]
    return labels, groups

def hard_emulator_priority(device_labels, device_groups):
    if ("Emulator" in device_groups
        and "Real_Device" in device_groups
        and not (set(device_labels) & REAL_DEVICE_STRONG_LABELS)):
        device_groups = [g for g in device_groups if g != "Real_Device"]
        device_labels = [l for l in device_labels if l not in REAL_DEVICE_GENERIC_ADB]
    return device_labels, device_groups

# --- Audit helpers (v2) ---
SUSPECT_LINE_PAT = re.compile(
    r'(?mi)^\s*[^\n]*\b(?:'
    r'(?:\./|\.\\)?gradle(?:w)?(?:\.bat)?|'
    r'am\s+instrument|'
    r'gcloud(?:\s+beta)?\s+firebase\s+test\s+android\s+run|'
    r'flank\s+android\s+run|'
    r'marathon\b|spoon\b'
    r')[^\n]*$'
)

def audit_reasons(text: str) -> tuple[list[str], list[str]]:
    reasons, lines = [], []
    for m in SUSPECT_LINE_PAT.finditer(text):
        ln = m.group(0).strip()
        if ln:
            lines.append(ln)
        if len(lines) >= 4:
            break
    # only flag if a tool-like line exists
    if re.search(r'(?mi)^\s*(script|run|command)\s*:\s*\|', text) and lines:
        reasons.append("gradle_in_run_block")
    if re.search(r'(?mi)^\s*(script|run|command)\s*:\s*(?!\|)\S', text) and lines:
        reasons.append("gradle_in_run_same_line")
    if re.search(r'(?mi)\bcd\s+\S+\s+&&\s+(?:\./|\.\\)?gradle', text) and lines:
        reasons.append("cd_and_chain")
    if re.search(r'(?mi)^\s*(?:\S+=\S+\s+)+(?:\./|\.\\)?gradle', text) and lines:
        reasons.append("env_prefix_or_abbr_cat")
    if not reasons and lines:
        reasons.append("unknown_trigger_shape")
    return reasons, lines

# === scan & export ===
rows: List[Dict[str, Any]] = []
miss_rows: List[Dict[str, Any]] = []

for fname in os.listdir(CONFIG_DIR):
    ext = os.path.splitext(fname)[1].lower()
    if ext not in ('.yml', '.yaml'):
        continue

    fpath = os.path.join(CONFIG_DIR, fname)
    if not os.path.isfile(fpath):
        continue

    with open(fpath, 'r', encoding='utf-8', errors='ignore') as f:
        raw = f.read()

    # ---------- Primary pass (fast) ----------
    content = strip_comments(raw)
    content = re.sub(r'(?m)^\s*-\s*', '', content)      # normalize bullets
    content = normalize_block_keys(content)             # flatten run/script/command

    device_labels, device_groups = collect_hits_with_groups(DEVICE_PATTERNS, content)
    trigger_labels, trigger_groups = collect_hits_with_groups(TRIGGER_PATTERNS, content)

    # gradle/gradle-build-action inputs -> treat as trigger
    if any_match(GHA_GRADLE_INPUTS, content):
        trigger_labels = unique_preserve(trigger_labels + ["gha gradle arguments"])
        trigger_groups = unique_preserve(trigger_groups + ["Gradle"])

    # Variable-hinted gradle
    for line in content.splitlines():
        if re.search(r'(?i)\bgradle(?:w)?(?:\.bat)?\b', line) and variable_hints_connected(line):
            trigger_labels = unique_preserve(trigger_labels + ["gradle via variable hint"])
            trigger_groups = unique_preserve(trigger_groups + ["Gradle"])
            break

    # WEAK-HINT GATING + RECONCILE
    device_labels, device_groups = filter_weak_device_hints(device_labels, device_groups)
    device_labels, device_groups = reconcile_emulator_vs_real(device_labels, device_groups)
    device_labels, device_groups = hard_emulator_priority(device_labels, device_groups)

    # ---------- Fallback (only if BOTH empty) ----------
    fallback_detected = False
    if not device_labels and not trigger_labels:
        fallback = strip_comments(raw)
        fallback = re.sub(r'(?m)^\s*-\s*', '', fallback)
        fallback = re.sub(r'(?mi)^\s*(?:command|run|script)\s*:\s*\|?\s*', '', fallback)
        fallback = re.sub(r'(?m)^\s*sudo\s+', '', fallback)

        fb_device_labels, fb_device_groups = collect_hits_with_groups(DEVICE_PATTERNS, fallback)
        fb_trigger_labels, fb_trigger_groups = collect_hits_with_groups(TRIGGER_PATTERNS, fallback)

        fb_device_labels, fb_device_groups = filter_weak_device_hints(fb_device_labels, fb_device_groups)
        fb_device_labels, fb_device_groups = reconcile_emulator_vs_real(fb_device_labels, fb_device_groups)

        if any_match(GHA_GRADLE_INPUTS, fallback):
            fb_trigger_labels.append("gha gradle arguments")
            fb_trigger_groups.append("Gradle")

        if fb_device_labels or fb_trigger_labels:
            fallback_detected = True
            device_labels  = unique_preserve(device_labels  + fb_device_labels)
            device_groups  = unique_preserve(device_groups  + fb_device_groups)
            trigger_labels = unique_preserve(trigger_labels + fb_trigger_labels)
            trigger_groups = unique_preserve(trigger_groups + fb_trigger_groups)
            device_labels, device_groups = hard_emulator_priority(device_labels, device_groups)

    # parse full_name and ci_platform from: <full_name>__<platform>++<file>.yml
    full_name = "Unknown"; ci_platform = "Unknown"
    base = os.path.basename(fname)
    if "__" in base and "++" in base:
        try:
            full_name = base.split("__", 1)[0]
            ci_platform = base.split("__", 1)[1].split("++", 1)[0]
        except Exception:
            pass

    has_device_setup = bool(device_labels)
    has_test_trigger = bool(trigger_labels)

    real_device_groups = {"Emulator", "GMD", "Third_Party_Lab", "Real_Device"}
    has_real_device_group = any(g in real_device_groups for g in device_groups)
    instru_t_ci = bool(has_test_trigger or (has_device_setup and has_real_device_group))

    miss_reasons_list, miss_lines = [], []
    if has_device_setup and not has_test_trigger:
        miss_reasons_list, miss_lines = audit_reasons(raw)

    row = {
        "filename": fname,
        "full_name": full_name,
        "ci_platform": ci_platform,
        "has_device_setup": has_device_setup,
        "device_setup": ", ".join(device_labels),
        "device_setup_group": ", ".join(device_groups),
        "has_test_trigger": has_test_trigger,
        "test_trigger": ", ".join(trigger_labels),
        "test_trigger_group": ", ".join(trigger_groups),
        "instru_t_ci": instru_t_ci,
        "Search_Method_Name": Search_Method_Name,
        "fall_back": bool(fallback_detected),
        "miss_reasons": ";".join(miss_reasons_list),
        "miss_evidence": " || ".join(miss_lines[:3]),
    }
    rows.append(row)
    if has_device_setup and not has_test_trigger:
        miss_rows.append(row)

# Export main CSV
df = pd.DataFrame(rows, columns=[
    "filename", "full_name", "ci_platform",
    "has_device_setup", "device_setup", "device_setup_group",
    "has_test_trigger", "test_trigger", "test_trigger_group",
    "instru_t_ci", "Search_Method_Name", "fall_back",
    "miss_reasons", "miss_evidence",
])
df.to_csv(OUTPUT_CSV, index=False)

# Export misses audit CSV + summary
if miss_rows:
    pd.DataFrame(miss_rows).to_csv(MISSES_CSV, index=False)
    reason_counter: Dict[str, int] = {}
    for r in miss_rows:
        if r["miss_reasons"]:
            for code in r["miss_reasons"].split(";"):
                reason_counter[code] = reason_counter.get(code, 0) + 1
        else:
            reason_counter["(none)"] = reason_counter.get("(none)", 0) + 1
    pd.DataFrame(
        sorted(reason_counter.items(), key=lambda kv: (-kv[1], kv[0])),
        columns=["reason_code", "count"]
    ).to_csv(MISSES_SUMMARY_CSV, index=False)

print(f"✅ Saved main CSV: {OUTPUT_CSV} (yaml files={len(df)})")
if miss_rows:
    print(f"⚠️  Misses audit CSV: {MISSES_CSV} (missed={len(miss_rows)})")
    print(f"📊 Miss reason summary: {MISSES_SUMMARY_CSV}")
else:
    print("🎉 No emulator-without-trigger misses detected.")


C:\Users\gilla\AppData\Local\Temp\ipykernel_15820\2791292007.py:24: DeprecationWarning: Flags not at the start of the expression '(?m)^\\s*\\S*sdkmanage' (truncated)
  return [re.compile(p, flags) for p in patterns]


✅ Saved main CSV: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\All_YMLs\3.1_YML_Files.csv (yaml files=12667)
⚠️  Misses audit CSV: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\All_YMLs\3.1_YML_Missed_Triggers_Audit.csv (missed=224)
📊 Miss reason summary: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\All_YMLs\3.1_YML_Missed_Triggers_Summary.csv


In [3]:
# Stage 2: Analyzing the leftovers for instru testing (fast, guarded)
# -*- coding: utf-8 -*-
import os, re, time
import pandas as pd
from typing import List, Pattern, Tuple, Dict

# ====== Time & size guards ====================================================
FILE_WATCHDOG_SEC   = 450       # max wall time per file
MAX_SCAN_CHARS      = 1000_000  # cap text scanned by heavy regex
MAX_RUNBLOCK_CHARS  = 1000_000  # cap extracted run/script text
# ============================================================================

# ==== CONFIG (edit these two) ================================================
CONFIG_DIR  = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files"
OUTPUT_DIR  = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\All_YMLs"
# ============================================================================
os.makedirs(OUTPUT_DIR, exist_ok=True)

LEFTOVERS_CSV       = os.path.join(OUTPUT_DIR, "3.1_YML_Missed_Triggers_Audit.csv")
FAST_CSV            = os.path.join(OUTPUT_DIR, "3.1_YML_Files_FAST.csv")

DEEP_CSV            = os.path.join(OUTPUT_DIR, "3.1_YML_Files_DEEP.csv")
DEEP_MISSES_CSV     = os.path.join(OUTPUT_DIR, "3.1_YML_Missed_Triggers_Audit_DEEP.csv")
DEEP_MISSES_SUMMARY = os.path.join(OUTPUT_DIR, "3.1_YML_Missed_Triggers_Summary_DEEP.csv")
DEEP_DEBUG_SNIPS    = os.path.join(OUTPUT_DIR, "3.1_YML_Debug_Snippets_DEEP.csv")

FINAL_MERGED_CSV    = os.path.join(OUTPUT_DIR, "3.1_YML_Files_FINAL.csv")
FINAL_MISSES_CSV    = os.path.join(OUTPUT_DIR, "3.1_YML_Missed_Triggers_Audit_FINAL.csv")
FINAL_MISSES_SUM    = os.path.join(OUTPUT_DIR, "3.1_YML_Missed_Triggers_Summary_FINAL.csv")

Search_Method_Name = (
    "Stage-2 DEEP — YAML-less run/script extractor + candidate-line filter + watchdog + "
    "broader trigger shapes + chains + managedDevices + 3P runners + Bazel/Buck/Fastlane/Make/NPM"
)

# ===== Progress display controls =====
PROGRESS_EVERY = 5   # print a full progress line every N files (set to 1 for every file)

# --- helpers -----------------------------------------------------------------
def compile_any(patterns: List[str], flags=re.I | re.M) -> List[Pattern]:
    return [re.compile(p, flags) for p in patterns]

def any_match(patterns: List[Pattern], text: str) -> bool:
    return any(p.search(text) for p in patterns)

def unique_preserve(seq: List[str]) -> List[str]:
    seen, out = set(), []
    for x in seq:
        if x not in seen:
            seen.add(x); out.append(x)
    return out

COMMENT_LINE_RE = re.compile(r'(?m)^\s*(#|//|REM\b|::).*?$')

def strip_comments(raw: str) -> str:
    return COMMENT_LINE_RE.sub("", raw or "")

def drop_block_keys(text: str) -> str:
    # remove header line only; keep the block body
    return re.sub(r'(?mi)^\s*(script|run|command)\s*:\s*(\|?>)?\s*$', '', text)

def flatten_backslashes(s: str) -> str:
    # join shell line continuations
    return re.sub(r'\\\r?\n\s*', ' ', s)

def normalize_for_scan(raw: str) -> str:
    txt = strip_comments(raw)
    txt = re.sub(r'(?m)^\s*-\s*', '', txt)   # bullet normalization
    txt = drop_block_keys(txt)
    txt = flatten_backslashes(txt)
    # trim extremely large inputs (safety)
    if len(txt) > MAX_SCAN_CHARS:
        txt = txt[:MAX_SCAN_CHARS]
    return txt

# === Fast, YAML-less run/script extractor (handles single-line & block forms) ===
RUN_HDR = re.compile(r'(?mi)^\s*(run|script|command)\s*:\s*(\|?>)?\s*(.*)$')
WITH_KEYS = {"script", "args", "command"}

def extract_runs_from_yaml(raw: str) -> List[str]:
    """
    Grab single-line 'run: ...' and block bodies after 'run: |' or 'run: >',
    plus 'with: { script/args/command }' forms—without parsing YAML.
    """
    out: List[str] = []
    lines = raw.replace('\t','  ').splitlines()
    N = len(lines)
    i = 0

    def read_block(start_idx: int, base_indent: int) -> str:
        j = start_idx
        chunks = []
        while j < N:
            line = lines[j]
            if not line.strip():
                chunks.append(""); j += 1; continue
            indent = len(line) - len(line.lstrip(' '))
            if indent <= base_indent:
                break
            chunks.append(line[base_indent:])
            if sum(len(c) + 1 for c in chunks) > MAX_RUNBLOCK_CHARS:
                break
            j += 1
        return "\n".join(chunks), j

    while i < N:
        m = RUN_HDR.match(lines[i])
        if m:
            _, block_mark, tail = m.group(1).lower(), m.group(2), m.group(3)
            if block_mark:  # run: |  or  run: >
                base_indent = len(lines[i]) - len(lines[i].lstrip(' '))
                block, j = read_block(i + 1, base_indent)
                if block:
                    out.append(block)
                i = j
                continue
            else:
                if tail:
                    out.append(tail)
            i += 1
            continue

        # naive 'with:' helper to catch short maps nearby
        if "with:" in lines[i].lower():
            j = i + 1
            base_indent = len(lines[i]) - len(lines[i].lstrip(' '))
            while j < min(N, i + 30):
                wline = lines[j]
                if (len(wline) - len(wline.lstrip(' '))) <= base_indent:
                    break
                wl = wline.strip().lower()
                if any(wl.startswith(k + ":") for k in WITH_KEYS):
                    # value after colon (same line)
                    val = wline.split(":", 1)[1].strip()
                    if val:
                        out.append(val)
                    # if block follows, read it
                    if wl.endswith("|") or wl.endswith(">"):
                        block, j2 = read_block(j + 1, len(wline) - len(wline.lstrip(' ')))
                        if block:
                            out.append(block)
                        j = j2 - 1
                j += 1

        i += 1

    return [flatten_backslashes(s)[:MAX_RUNBLOCK_CHARS] for s in out if s]

# === Candidate-line filter to avoid heavy regex on whole files ================
SUSPECT_TOKENS = (
    "gradle", "gradlew", "am instrument",
    "gcloud firebase test android run", "flank android run",
    "marathon", "spoon", "bazel", "buck", "fastlane",
    "npm run", "yarn", "pnpm run", "make",
    "android-wait-for-emulator", "emulator", "avdmanager", "sdkmanager"
)

def filter_suspect_text(full_text: str) -> str:
    lines = full_text.splitlines()
    keep = [ln for ln in lines if any(tok in ln.lower() for tok in SUSPECT_TOKENS)]
    joined = "\n".join(keep)
    return joined[:MAX_SCAN_CHARS] if len(joined) > MAX_SCAN_CHARS else joined

# --------- patterns (deep set) -----------------------------------------------
DEVICE_SOURCES: List[Tuple[str, str, List[str]]] = [
    ("Emulator", "reactivecircus runner", [r'uses:\s*reactivecircus/android-emulator-runner']),
    ("Emulator", "android-wait-for-emulator", [r'(?m)^\s*(?:\./)?android-wait-for-emulator\b']),
    ("Emulator", "emulator -avd/@", [r'(?m)^\s*\S*emulator\b[^\n]*\s(-avd|@)\S+']),
    ("Emulator", "avdmanager", [r'(?m)^\s*\S*avdmanager\b']),
    ("Emulator", "sdkmanager system-images/emulator", [
        r'(?m)^\s*\S*sdkmanager\b[^\n"]*"system-images;android-(?:\d+|\$[A-Z_][A-Z0-9_]*)[^"\n]*"|'
        r'(?m)^\s*\S*sdkmanager\b[^\n]*\bsystem-images;android-(?:\d+|\$[A-Z_][A-Z0-9_]*)\b'
    ]),
    ("Emulator", "adb -s emulator-serial", [
        r'(?m)^\s*adb\s+-s\s+emulator-\d+\b',
        r'(?m)^\s*adb\s+-s\s+(?:localhost|127\.0\.0\.1):\d+\b',
    ]),
    ("GMD", "managedDevices DSL", [r'\bmanageddevices?\b']),
    ("GMD", "ManagedVirtualDevice DSL", [r'\bmanagedvirtualdevice\b|\bcom\.android\.build\.api\.dsl\.ManagedVirtualDevice\b']),
    ("GMD", "GMD task mentions", [r'\bmanageddevice\w*androidtest\b']),
]

GRADLE_PREFIX = (
    r'^\s*'
    r'(?:\S+=\S+\s+)*'
    r'(?:sudo\s+)?'
    r'(?:(?:bash|sh)\s+-c[l]?\s+[\'"]?)?'
    r'(?:[^#\n]*?(?:;|&&|\|\|)\s+)*'
    r'(?:cd\s+\S+\s+&&\s+)?'
    r'(?:\./|\.\\)?gradle(?:w)?(?:\.bat)?'
)

GRADLE_ANYWHERE = r'(?i)[^\n]*\bgradle(?:w)?(?:\.bat)?[^\n]*'
NON_TEST_PREFIX = r'(?:assemble|bundle|package|compile|merge|process|generate|install|uninstall|jacoco|lint|publish|sign|upload|shadow|kover)'

TRIGGER_SOURCES: List[Tuple[str, str, List[str]]] = [
    ("Gradle", "connectedAndroidTest",                 [rf'(?mi){GRADLE_PREFIX}[^\n]*\bconnectedandroidtest\b']),
    ("Gradle", "connected.*Android.*",                 [rf'(?mi){GRADLE_PREFIX}[^\n]*\bconnected[\w:.-]*android[\w:.-]*test\b']),
    ("Gradle", "connectedCheck",                       [rf'(?mi){GRADLE_PREFIX}\s+(?::[\w-]+:)*connectedcheck\b']),
    ("Gradle", "plain androidTest",                    [rf'(?mi){GRADLE_PREFIX}[^\n]*\b(?::[\w-]+:)*androidtest\b']),
    ("Gradle", "cAT (abbr)",                           [rf'(?mi){GRADLE_PREFIX}[^\n]*\b(?::[\w-]+:)*cat\b']),
    ("Gradle", "deviceCheck",                          [rf'(?mi){GRADLE_PREFIX}\s+(?::[\w-]+:)*(?:devicecheck|alldevicechecks)\b']),
    ("Gradle", "managedDevice AndroidTest",            [rf'(?mi){GRADLE_PREFIX}[^\n]*\b(?!{NON_TEST_PREFIX})(?:manageddevice|device)[\w:-]*androidtest\b']),

    ("Gradle", "connected (anywhere)",                 [rf'(?mi){GRADLE_ANYWHERE}\bconnected[\w:.-]*android[\w:.-]*test\b']),
    ("Gradle", "cAT (abbr, anywhere)",                 [rf'(?mi){GRADLE_ANYWHERE}\b(?::[\w-]+:)*cat\b']),
    ("Gradle", "deviceCheck (anywhere)",               [rf'(?mi){GRADLE_ANYWHERE}\b(?::[\w-]+:)*(?:devicecheck|alldevicechecks)\b']),
    ("Gradle", "managedDevice AndroidTest (anywhere)", [rf'(?mi){GRADLE_ANYWHERE}\b(?!{NON_TEST_PREFIX})(?:manageddevice|device)[\w:-]*androidtest\b']),
    ("Gradle", "plain AndroidTest (anywhere)",         [rf'(?mi){GRADLE_ANYWHERE}\b(?::[\w-]+:)*androidtest\b']),

    ("ADB", "am instrument",                           [r'(?mi)^[^\n]*\bam\s+instrument\b']),
    ("Third_Party_Lab", "gcloud firebase",             [r'(?mi)^[^\n]*\bgcloud(?:\s+beta)?\s+firebase\s+test\s+android\s+run\b']),
    ("Third_Party_Lab", "Flank",                       [r'(?mi)^[^\n]*\bflank\s+android\s+run\b']),
    ("Gradle", "Spoon/Marathon",                       [r'(?mi)^[^\n]*\b(spoon|marathon)\b']),
    ("Bazel", "bazel mobile-install/test",             [r'(?mi)^[^\n]*\bbazel\s+(?:mobile-install|test)\b[^\n]*(android_instrumentation_test|instrument.*android.*test)\b']),
    ("Buck", "buck test",                              [r'(?mi)^[^\n]*\bbuck\s+test\b[^\n]*(android|instrument)']),
    ("Fastlane", "fastlane android ui/instrument",     [r'(?mi)^[^\n]*\b(?:bundle\s+exec\s+)?fastlane\b[^\n]*(connected|instrument|espresso|ui[_-]?tests?|androidtest)\b']),
    ("Make", "make *AndroidTest/deviceCheck",          [r'(?mi)^[^\n]*\bmake\b[^\n]*(connected(?:android)?test|androidtest|device(?:check|tests?)|instrument(?:ation)?tests?)\b']),
    ("NPM", "npm/yarn/pnpm run *androidtest/e2e",      [r'(?mi)^[^\n]*\b(?:npm\s+run|yarn\s+run|yarn|pnpm\s+run)\b[^\n]*(androidtest|connected|espresso|instrument|e2e)\b']),
]

GHA_GRADLE_INPUTS = compile_any([
    r'(?mi)^\s*arguments\s*:\s*(?::[\w-]+:)*connectedcheck\b',
    r'(?mi)^\s*arguments\s*:\s*[:\w-]*connected.*android.*test\b',
    r'(?mi)^\s*arguments\s*:\s*\bcat\b',
    r'(?mi)^\s*arguments\s*:\s*(?::[\w-]+:)*(?:devicecheck|alldevicechecks)\b',
    r'(?mi)^\s*tasks?\s*:\s*(?::[\w-]+:)*connectedcheck\b',
    r'(?mi)^\s*tasks?\s*:\s*[:\w-]*connected.*android.*test\b',
    r'(?mi)^\s*tasks?\s*:\s*(?::[\w-]+:)*(?:devicecheck|alldevicechecks)\b',
    r'(?mi)^\s*tasks?\s*:\s*[\w:-]*androidtest\b',
])

def collect_hits_with_groups(patterns, text: str):
    """
    patterns: List[Tuple[group_name, label, List[Union[str, Pattern]]]]
    Returns: (labels, groups, evidence) preserving source order with de-dup
    """
    labels, groups, evidence = [], [], []
    for grp, lbl, pats in patterns:
        if isinstance(pats, str):
            pats = [pats]
        for p in pats:
            rx = re.compile(p, re.I | re.M) if isinstance(p, str) else p
            m = rx.search(text)
            if m:
                labels.append(lbl); groups.append(grp)
                line_start = text.rfind("\n", 0, m.start()) + 1
                line_end = text.find("\n", m.end());  line_end = len(text) if line_end == -1 else line_end
                evidence.append(text[line_start:line_end].strip()[:300])
                break
    return unique_preserve(labels), unique_preserve(groups), unique_preserve(evidence)

EMULATOR_STRONG = {"reactivecircus runner","android-wait-for-emulator","emulator -avd/@","avdmanager","sdkmanager system-images/emulator","adb -s emulator-serial"}
REAL_DEVICE_STRONG = {"adb get-state","adb get-serialno"}
REAL_DEVICE_GENERIC = {"adb devices","adb install", "adb shell", "adb root", "adb settings", "adb input", "adb pm grant"}

def reconcile(labels, groups):
    lbls = set(labels)
    if lbls & EMULATOR_STRONG and "Real_Device" in groups and not (lbls & REAL_DEVICE_STRONG):
        groups = [g for g in groups if g != "Real_Device"]
        labels = [l for l in labels if l not in REAL_DEVICE_GENERIC]
    return labels, groups

SUSPECT_LINE_PAT = re.compile(
    r'(?mi)^[^\n]*\b('
    r'(?:\./|\.\\)?gradle(?:w)?(?:\.bat)?|'
    r'am\s+instrument|'
    r'gcloud(?:\s+beta)?\s+firebase\s+test\s+android\s+run|'
    r'flank\s+android\s+run|'
    r'marathon\b|spoon\b|'
    r'bazel\s+(?:mobile-install|test)|'
    r'buck\s+test|'
    r'(?:bundle\s+exec\s+)?fastlane\b|'
    r'(?:npm\s+run|yarn\s+run|yarn|pnpm\s+run)|'
    r'make\b'
    r')[^\n]*$'
)

def audit_reasons(raw: str, normalized: str) -> tuple[list[str], list[str]]:
    reasons, lines = [], []
    for m in SUSPECT_LINE_PAT.finditer(raw):
        ln = m.group(0).strip()
        if ln:
            lines.append(ln)
        if len(lines) >= 6: break
    if re.search(r'(?mi)^\s*(script|run|command)\s*:\s*\|', raw): reasons.append("gradle_in_run_block")
    if re.search(r'(?mi)^\s*(script|run|command)\s*:\s*(?!\|)\S', raw): reasons.append("gradle_in_run_same_line")
    if re.search(r'(?mi)\bcd\s+\S+\s+&&\s+(?:\./|\.\\)?gradle', normalized): reasons.append("cd_and_chain")
    if re.search(r'(?mi)^\s*(?:\S+=\S+\s+)+(?:\./|\.\\)?gradle', normalized): reasons.append("env_prefix")
    if not reasons and lines: reasons.append("unknown_trigger_shape")
    return reasons, lines[:6]

# ---------- utilities for progress ----------
def fmt_dur(seconds: float) -> str:
    seconds = max(0, int(seconds))
    m, s = divmod(seconds, 60)
    h, m = divmod(m, 60)
    return f"{h:d}:{m:02d}:{s:02d}" if h else f"{m:02d}:{s:02d}"

def print_progress(i: int, n: int, start: float, found: int, misses: int, fname: str, force_newline=False):
    elapsed = time.time() - start
    rate = (i / elapsed) if elapsed > 0 else 0.0
    remain = (n - i) / rate if rate > 0 else 0.0
    pct = (i / n) * 100 if n else 100.0
    line = (f"[DEEP] {i:>4}/{n:<4} ({pct:5.1f}%)  "
            f"elapsed {fmt_dur(elapsed)}  eta {fmt_dur(remain)}  "
            f"triggers:{found}  misses:{misses}  "
            f"{fname}")
    end = "\n" if force_newline else "\r"
    print(line[:200], end=end, flush=True)  # trim long filenames for neatness

# ----- load leftovers -----
if not os.path.exists(LEFTOVERS_CSV):
    raise SystemExit(f"Leftovers list not found: {LEFTOVERS_CSV}\nRun Stage-1 first.")

left_df = pd.read_csv(LEFTOVERS_CSV)
left_files = sorted(set(left_df["filename"].tolist()))
total = len(left_files)
print(f"DEEP scan on leftovers: {total} files")

rows, misses_rows, debug_rows = [], [], []
triggers_found = 0
misses_count = 0

start_ts = time.time()

for idx, fname in enumerate(left_files, 1):
    try:
        fpath = os.path.join(CONFIG_DIR, fname)
        if not os.path.isfile(fpath):
            misses_count += 1
            rows.append({
                "filename": fname, "full_name":"Unknown","ci_platform":"Unknown",
                "has_device_setup": False, "device_setup":"", "device_setup_group":"",
                "has_test_trigger": False, "test_trigger":"", "test_trigger_group":"",
                "instru_t_ci": False, "Search_Method_Name": Search_Method_Name,
                "fall_back": False, "miss_reasons":"file_missing", "miss_evidence":""
            })
            print_progress(idx, total, start_ts, triggers_found, misses_count, f"(missing) {fname}",
                           force_newline=(idx % PROGRESS_EVERY == 0 or idx == total))
            continue

        t0 = time.perf_counter()
        raw = open(fpath, "r", encoding="utf-8", errors="ignore").read()

        # 1) Normalize (guard)
        norm = normalize_for_scan(raw)
        if time.perf_counter() - t0 > FILE_WATCHDOG_SEC:
            rows.append({
                "filename": fname, "full_name":"Unknown","ci_platform":"Unknown",
                "has_device_setup": False, "device_setup":"", "device_setup_group":"",
                "has_test_trigger": False, "test_trigger":"", "test_trigger_group":"",
                "instru_t_ci": False, "Search_Method_Name": Search_Method_Name,
                "fall_back": False, "miss_reasons":"timeout_normalize", "miss_evidence":""
            })
            print_progress(idx, total, start_ts, triggers_found, misses_count+1, f"(timeout normalize) {fname}", True)
            continue

        # 2) Fast run/script extraction (no YAML)
        runs_text = "\n".join(extract_runs_from_yaml(raw))

        # 3) Pre-filter lines to avoid catastrophic backtracking
        suspect_norm = filter_suspect_text(norm)
        suspect_runs = filter_suspect_text(runs_text) if runs_text else ""

        # 4) Devices first (cheap)
        dev_labels, dev_groups, dev_evidence = collect_hits_with_groups(DEVICE_SOURCES, suspect_norm.lower())

        # 5) Triggers – prefer run/script blocks first
        trg2_lbl, trg2_grp, trg2_evd = collect_hits_with_groups(TRIGGER_SOURCES, suspect_runs)
        if trg2_lbl:
            trg1_lbl, trg1_grp, trg1_evd = [], [], []
        else:
            # still nothing—only now pay the cost of full suspect_norm
            if time.perf_counter() - t0 > FILE_WATCHDOG_SEC:
                rows.append({
                    "filename": fname, "full_name":"Unknown","ci_platform":"Unknown",
                    "has_device_setup": bool(dev_labels), "device_setup": ", ".join(dev_labels),
                    "device_setup_group": ", ".join(dev_groups),
                    "has_test_trigger": False, "test_trigger":"", "test_trigger_group":"",
                    "instru_t_ci": False, "Search_Method_Name": Search_Method_Name,
                    "fall_back": False, "miss_reasons":"timeout_before_trigger", "miss_evidence":""
                })
                print_progress(idx, total, start_ts, triggers_found, misses_count+1, f"(timeout pre-trigger) {fname}", True)
                continue
            trg1_lbl, trg1_grp, trg1_evd = collect_hits_with_groups(TRIGGER_SOURCES, suspect_norm)

        # 6) Combine triggers
        trg_labels   = unique_preserve(trg1_lbl + trg2_lbl)
        trg_groups   = unique_preserve(trg1_grp + trg2_grp)
        trg_evidence = unique_preserve(trg1_evd + trg2_evd)

        # 7) GHA gradle inputs on suspect_norm
        if any_match(GHA_GRADLE_INPUTS, suspect_norm):
            trg_labels   = unique_preserve(trg_labels + ["gha gradle arguments"])
            trg_groups   = unique_preserve(trg_groups + ["Gradle"])
            trg_evidence = unique_preserve(trg_evidence + ["(gha gradle inputs)"])

        # 8) Reconcile emulator vs real
        dev_labels, dev_groups = reconcile(dev_labels, dev_groups)

        # 9) Pull metadata from filename
        full_name = "Unknown"; ci_platform = "Unknown"
        base = os.path.basename(fname)
        if "__" in base and "++" in base:
            try:
                full_name = base.split("__", 1)[0]
                ci_platform = base.split("__", 1)[1].split("++", 1)[0]
            except Exception:
                pass

        has_device  = bool(dev_labels)
        has_trigger = bool(trg_labels)
        real_groups = {"Emulator","GMD","Third_Party_Lab","Real_Device"}
        instru_t_ci = bool(has_trigger or (has_device and any(g in real_groups for g in dev_groups)))

        # 10) Watchdog final check (record partials if over budget)
        if time.perf_counter() - t0 > FILE_WATCHDOG_SEC:
            miss_code = "timeout_after_trigger_found" if has_trigger else "timeout_after_scan"
            rows.append({
                "filename": fname,
                "full_name": full_name, "ci_platform": ci_platform,
                "has_device_setup": has_device,
                "device_setup": ", ".join(dev_labels),
                "device_setup_group": ", ".join(dev_groups),
                "has_test_trigger": has_trigger,
                "test_trigger": ", ".join(trg_labels),
                "test_trigger_group": ", ".join(trg_groups),
                "instru_t_ci": instru_t_ci,
                "Search_Method_Name": Search_Method_Name,
                "fall_back": False,
                "miss_reasons": miss_code,
                "miss_evidence": (trg_evidence[0] if trg_evidence else "")
            })
            print_progress(idx, total, start_ts, triggers_found + int(has_trigger), misses_count + int(has_device and not has_trigger), f"(timeout final) {fname}", True)
            continue

        # 11) Miss audit (only when emulator/device but no trigger)
        miss_reasons, miss_lines = ([], [])
        if has_device and not has_trigger:
            miss_reasons, miss_lines = audit_reasons(raw, norm)
            misses_count += 1
        if has_trigger:
            triggers_found += 1
            if trg_evidence:
                debug_rows.append({
                    "filename": fname,
                    "first_trigger_evidence": trg_evidence[0],
                    "all_trigger_labels": ", ".join(trg_labels)
                })

        row = {
            "filename": fname,
            "full_name": full_name,
            "ci_platform": ci_platform,
            "has_device_setup": has_device,
            "device_setup": ", ".join(dev_labels),
            "device_setup_group": ", ".join(dev_groups),
            "has_test_trigger": has_trigger,
            "test_trigger": ", ".join(trg_labels),
            "test_trigger_group": ", ".join(trg_groups),
            "instru_t_ci": instru_t_ci,
            "Search_Method_Name": Search_Method_Name,
            "fall_back": False,
            "miss_reasons": ";".join(miss_reasons),
            "miss_evidence": " || ".join(miss_lines),
        }
        rows.append(row)
        if has_device and not has_trigger:
            misses_rows.append(row)

        # progress output
        print_progress(idx, total, start_ts, triggers_found, misses_count, fname,
                       force_newline=(idx % PROGRESS_EVERY == 0 or idx == total))

    except Exception as e:
        # Never crash the whole run; record and move on
        misses_count += 1
        rows.append({
            "filename": fname, "full_name":"Unknown","ci_platform":"Unknown",
            "has_device_setup": False, "device_setup":"", "device_setup_group":"",
            "has_test_trigger": False, "test_trigger":"", "test_trigger_group":"",
            "instru_t_ci": False, "Search_Method_Name": Search_Method_Name,
            "fall_back": False, "miss_reasons": f"exception:{type(e).__name__}", "miss_evidence": str(e)[:300]
        })
        print_progress(idx, total, start_ts, triggers_found, misses_count, f"(exception) {fname}", True)
        continue

# export deep results
deep_df = pd.DataFrame(rows)
deep_df.to_csv(DEEP_CSV, index=False)
print(f"\n✅ DEEP per-file CSV: {DEEP_CSV} (rows={len(deep_df)})")

if misses_rows:
    pd.DataFrame(misses_rows).to_csv(DEEP_MISSES_CSV, index=False)
    reason_counter: Dict[str,int] = {}
    for r in misses_rows:
        codes = [c for c in (r.get("miss_reasons") or "").split(";") if c] or ["(none)"]
        for c in codes:
            reason_counter[c] = reason_counter.get(c, 0) + 1
    pd.DataFrame(sorted(reason_counter.items(), key=lambda kv: (-kv[1], kv[0])),
                 columns=["reason_code","count"]).to_csv(DEEP_MISSES_SUMMARY, index=False)
    print(f"⚠️  DEEP misses audit: {DEEP_MISSES_CSV}")
    print(f"📊 DEEP miss summary: {DEEP_MISSES_SUMMARY}")

if debug_rows:
    pd.DataFrame(debug_rows).to_csv(DEEP_DEBUG_SNIPS, index=False)
    print(f"🔎 DEEP trigger snippets: {DEEP_DEBUG_SNIPS}")

# ---------- MERGE with Stage-1 ----------
fast_df = pd.read_csv(FAST_CSV) if os.path.exists(FAST_CSV) else pd.DataFrame(columns=deep_df.columns)
if not fast_df.empty:
    keep_fast = fast_df[~fast_df["filename"].isin(deep_df["filename"])]
    final = pd.concat([keep_fast, deep_df], ignore_index=True)
else:
    final = deep_df.copy()

final.to_csv(FINAL_MERGED_CSV, index=False)
print(f"🧩 Final merged CSV: {FINAL_MERGED_CSV} (rows={len(final)})")

# Merge audits (FAST + DEEP) for a single view
fast_miss_path = os.path.join(OUTPUT_DIR, "3.1_YML_Missed_Triggers_Audit_FAST.csv")
fast_miss = pd.read_csv(fast_miss_path) if os.path.exists(fast_miss_path) else pd.DataFrame()
deep_miss = pd.read_csv(DEEP_MISSES_CSV) if os.path.exists(DEEP_MISSES_CSV) else pd.DataFrame()

if not fast_miss.empty or not deep_miss.empty:
    both = pd.concat([fast_miss, deep_miss], ignore_index=True).drop_duplicates(subset=["filename"], keep="last")
    both.to_csv(FINAL_MISSES_CSV, index=False)
    # summary
    summary: Dict[str,int] = {}
    for _, r in both.iterrows():
        codes = [c for c in str(r.get("miss_reasons") or "").split(";") if c] or ["(none)"]
        for c in codes:
            summary[c] = summary.get(c, 0) + 1
    pd.DataFrame(sorted(summary.items(), key=lambda kv: (-kv[1], kv[0])),
                 columns=["reason_code","count"]).to_csv(FINAL_MISSES_SUM, index=False)
    print(f"📦 Final misses audit: {FINAL_MISSES_CSV}")
    print(f"📊 Final misses summary: {FINAL_MISSES_SUM}")
else:
    print("🎉 No outstanding misses to merge.")


DEEP scan on leftovers: 224 files
[DEEP]    5/224  (  2.2%)  elapsed 00:00  eta 00:01  triggers:0  misses:5  admob-plus.admob-plus__github_actions++ci.ymlandroid_ui_tests.ymlml
[DEEP]   10/224  (  4.5%)  elapsed 00:00  eta 00:01  triggers:0  misses:9  android.nowinandroid__github_actions++release.ymllineprofiles.yaml
[DEEP]   15/224  (  6.7%)  elapsed 00:00  eta 00:01  triggers:0  misses:11  ardriveapp.ardrive-web__github_actions++pr.yamlmllator.yml
[DEEP]   20/224  (  8.9%)  elapsed 00:00  eta 00:01  triggers:1  misses:12  aws.amazon-ivs-react-native-player__github_actions++pr.yaml
[DEEP]   25/224  ( 11.2%)  elapsed 00:00  eta 00:00  triggers:2  misses:13  canopas.cloud-gallery__github_actions++analyze.ymlse-apk.yml.yml
[DEEP]   30/224  ( 13.4%)  elapsed 00:00  eta 00:00  triggers:2  misses:13  canopas.khelo__github_actions++function_deploy.ymlyml.yml
[DEEP]   35/224  ( 15.6%)  elapsed 00:00  eta 00:00  triggers:2  misses:16  cliqz-oss.browser-android__azure_pipelines++azure-pipelines